# Normalización del Modelo — Tercera Forma Normal (3NF)

| | |
|---|---|
| **Proyecto** | Team Challenge SQL — E-commerce de tecnología |
| **Rol** | Data Modeler |
| **Rama** | `feature/er-diagram` |

---

## Introducción

El modelo relacional diseñado para este e-commerce está normalizado hasta la **Tercera Forma Normal (3NF)**.
A continuación se justifica el cumplimiento de cada forma normal con ejemplos concretos extraídos del modelo.


---

## 1️⃣ 1NF — Primera Forma Normal

> **Regla:** todos los atributos deben ser **atómicos** (un único valor por celda) y no debe haber grupos repetidos ni listas dentro de un campo.

### Cumplimiento en el modelo

Todas las columnas del modelo almacenan un único valor atómico:

| Tabla | Columna | ¿Es atómica? | Explicación |
|---|---|---|---|
| `customers` | `acquisition_channel` | ✅ Sí | Un único valor: `organic`, `paid_ads`, `social`… no una lista |
| `orders` | `status` | ✅ Sí | Un único estado a la vez: `pending`, `shipped`, `delivered`… |
| `payments` | `status` | ✅ Sí | Un único estado: `completed`, `failed`, `pending`… |
| `reviews` | `rating` | ✅ Sí | Un único entero entre 1 y 5 |
| ~~`orders`~~ | ~~`producto_1, producto_2`~~ | ❌ No aplicado | Grupos repetidos — **evitado** con `order_items` |

Los productos de un pedido **no** se almacenan en columnas `producto_1`, `producto_2`…  
Cada producto ocupa su propia fila en `order_items`. Eso es exactamente lo que exige 1NF.

**✅ El modelo cumple 1NF.**


### ❌ Ejemplo de lo que NO hacemos (viola 1NF)

```
orders
+-----------+------------------+------------------+
| order_id  | producto_1       | producto_2       |
+-----------+------------------+------------------+
| 1         | iPhone 15 (x2)   | AirPods Pro (x1) |  ← grupos repetidos
+-----------+------------------+------------------+
```

### ✅ Lo que SÍ hacemos (cumple 1NF)

```
order_items
+--------------+-----------+------------+----------+------------+
| order_item_id| order_id  | product_id | quantity | unit_price |
+--------------+-----------+------------+----------+------------+
| 1            | 1         | 101        | 2        | 999.00     |  ← iPhone 15
| 2            | 1         | 205        | 1        | 249.00     |  ← AirPods Pro
+--------------+-----------+------------+----------+------------+
```


---

## 2️⃣ 2NF — Segunda Forma Normal

> **Regla:** debe cumplir 1NF y todos los atributos no-clave deben depender de la **clave primaria completa**, no solo de una parte de ella.  
> Esta regla es relevante cuando la clave primaria es **compuesta**.

### Caso crítico: tabla `order_items`

`order_items` relaciona pedidos con productos. Si su PK fuera compuesta `(order_id, product_id)`,  
debería verificarse que cada atributo depende de **ambas columnas juntas**:

| Atributo | ¿De qué depende? | ¿Viola 2NF? |
|---|---|---|
| `quantity` | De `(order_id + product_id)` — unidades de ese producto en ese pedido | ✅ No |
| `unit_price` | De `(order_id + product_id)` — precio de ese producto en ese pedido concreto | ✅ No |
| `discount` | De `(order_id + product_id)` — descuento en esa línea concreta | ✅ No |
| ~~`product_name`~~ | Solo de `product_id` — el nombre no cambia por pedido | ❌ Violaría 2NF |

**Decisión de diseño:** `product_name`, `price`, `cost` y demás atributos del producto  
viven únicamente en la tabla `products`.  
En `order_items` solo guardamos los datos propios de esa línea: `quantity`, `unit_price`, `discount`.

Se usa `order_item_id` como **PK surrogate** (clave artificial autoincremental),  
lo que refuerza el cumplimiento de 2NF al evitar ambigüedades con claves compuestas.

**✅ El modelo cumple 2NF.**


### Visualización de la dependencia correcta

```
                    order_item_id (PK surrogate)
                          │
          ┌───────────────┼───────────────┐
          ▼               ▼               ▼
       quantity       unit_price       discount
    (depende de PK) (depende de PK) (depende de PK)


  product_name  ──►  product_id (FK)   ← vive en products, no aquí
```


---

## 3️⃣ 3NF — Tercera Forma Normal

> **Regla:** debe cumplir 2NF y **no debe haber dependencias transitivas**.  
> Ningún atributo no-clave puede depender de otro atributo no-clave — solo puede depender de la PK.

Una dependencia transitiva ocurre cuando: **`PK → A → B`**, siendo A un atributo no-clave.


### ❓ Pregunta 1: ¿Por qué `unit_price` está en `order_items` y no se lee de `products.price`?

`unit_price` en `order_items` almacena el **precio en el momento exacto de la compra**.  
Esto es necesario porque:

- `products.price` es el precio *actual* del producto, que puede cambiar en cualquier momento  
  (descuentos, actualizaciones de catálogo, inflación…).
- Si leyéramos siempre de `products.price`, el historial de ventas se corrompería  
  cada vez que un precio cambiara.
- `unit_price` depende de `order_item_id` (la PK), no genera dependencia transitiva alguna.

```
order_item_id ──► unit_price   ✅ dependencia directa con la PK
order_item_id ──► product_id ──► products.price   ← NO usamos esto para el histórico
```

**✅ Conclusión:** guardar `unit_price` en `order_items` es correcto tanto por integridad histórica como por normalización.


### ❓ Pregunta 2: ¿Por qué `country` está en `customers` y no en una tabla `countries` separada?

Crear una tabla `countries` separada solo estaría justificado si hubiera  
**otros atributos propios del país** que almacenar (por ejemplo: `currency`, `tax_rate`, `language`…).

En este modelo, `country` es simplemente un texto que identifica el origen geográfico del cliente.  
No hay más atributos asociados al país, por lo que no existe dependencia transitiva:

```
customer_id ──► country   ✅ dependencia directa con la PK
```

Si añadiéramos `currency` a `customers`, **sí** habría dependencia transitiva:

```
customer_id ──► country ──► currency   ❌ dependencia transitiva → necesitaría tabla countries
```

**✅ Conclusión:** con el alcance actual del modelo, `country` en `customers` no viola 3NF.  
Crear una tabla `countries` sería *over-engineering* innecesario.


### ❓ Pregunta 3: Si en `orders` almacenásemos `customer_name`, ¿qué forma normal se violaría?

Se violaría la **Tercera Forma Normal (3NF)** por **dependencia transitiva**.

La cadena de dependencia sería:

```
order_id ──► customer_id ──► customer_name
    PK         no-clave         no-clave
                   └─────────────────────► dependencia transitiva ❌
```

- `customer_name` **no** depende de `order_id` (la PK de orders)
- `customer_name` depende de `customer_id`, que es un atributo **no-clave** en `orders`
- Esto es exactamente la definición de dependencia transitiva

#### Problemas concretos que causaría:

| Anomalía | Descripción |
|---|---|
| **Actualización** | Si el cliente cambia su nombre, habría que actualizarlo en **todas** sus filas de `orders` |
| **Inserción** | No se podría insertar un pedido sin conocer el nombre del cliente |
| **Redundancia** | El nombre se repetiría en cada pedido que haya hecho ese cliente |

#### ✅ Solución aplicada

`customer_name` vive únicamente en `customers`.  
Para obtenerlo en una consulta se hace JOIN:


In [ ]:
# Ejemplo de query correcta — obtenemos customer_name con JOIN
query = '''
SELECT
  o.order_id,
  c.name  AS customer_name,
  o.status,
  o.ordered_at
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
ORDER BY o.ordered_at DESC
LIMIT 10
'''

# run_query(query)  # descomentar cuando BigQuery esté configurado
print('Query lista para ejecutar en BigQuery')


---

## 📋 Resumen de decisiones de diseño

| Decisión | Forma normal | Justificación |
|---|---|---|
| `unit_price` en `order_items` | 3NF | El precio al comprar puede cambiar; es un dato del evento, no del producto |
| `product_name` solo en `products` | 2NF | Dependería solo de `product_id`, no de la PK completa de `order_items` |
| `country` directo en `customers` | 3NF | Sin más atributos de país no hay dependencia transitiva |
| No guardar `customer_name` en `orders` | 3NF | Evita la cadena `order_id → customer_id → customer_name` |
| `reviews` apunta a `order_items` | Diseño | Se valora un producto específico comprado, no el pedido completo |
| PK surrogate en `order_items` | 2NF | Evita ambigüedad con claves compuestas |

---

**✅ El modelo cumple 1NF, 2NF y 3NF.**
